# SBV2-KR Google Colab 학습

Google Colab에서 SBV2-KR(Style-Bert-VITS2 한국어 포트)로 한국어 음성 모델을 학습합니다.

이 노트북은 기본적으로 Google Drive에 `SBV2-KR` 폴더를 만들고 그 안에서만 작업합니다. 다른 폴더는 건드리지 않습니다.
Google Drive를 쓰지 않는다면 1. 초기 설정에서 경로를 바꿔 주세요.

## 흐름

### 처음부터 학습할 때
위에서부터 차례로 실행하면 됩니다. 음성 합성에 필요한 파일은 Google Drive의 `SBV2-KR/model_assets/`에 저장됩니다. 중간 결과도 `SBV2-KR/Data/`에 저장되므로 학습을 중단했다가 이어서 할 수 있습니다.

### 학습을 이어서 할 때
0과 1을 실행한 뒤 3. 전처리는 건너뛰고 4부터 시작하세요. 학습 결과 확인과 스타일 나누기는 학습이 끝난 뒤 5에서 필요하면 진행하세요.


## 0. 환경 구축

Colab에 SBV2-KR 환경을 구축합니다. 실행하기 전에 「런타임 > 런타임 유형 변경」에서 다음을 확인하세요.
- **하드웨어 가속기**: T4 등 GPU
- **런타임 버전**: Python 3.12 이하(예: `2026.07`). Python 3.13 이미지에서는 `numpy<2` 등 의존성을 설치할 수 없습니다.

**참고**: 이 셀을 실행하면 세션이 다운되었다는 경고가 뜨지만 **무시하고 다음으로** 진행하세요. (설치한 패키지를 적용하려고 `exit()`로 런타임을 한 번 재시작하기 때문입니다.)


In [ ]:
!git clone https://github.com/ohgi07/SBV2-KR.git
%cd SBV2-KR/
!pip install -r requirements.txt
# requirements.txt의 torch<2.6과 맞지 않는 Colab 기본 torchvision을 제거 (남아 있으면 transformers 임포트가 실패)
!pip uninstall -y torchvision
!python initialize.py --skip_default_models

exit()

In [ ]:
# Google Drive를 쓴다면 실행하세요.

from google.colab import drive

drive.mount("/content/drive")

## 1. 초기 설정

학습 데이터와 결과를 저장할 디렉터리를 지정합니다.
Google Drive를 쓴다면 그대로 실행하고, 바꾸고 싶다면 수정한 뒤 실행하세요.


In [ ]:
# 작업 디렉터리로 이동
%cd /content/SBV2-KR/

# 학습에 필요한 파일과 중간 결과가 저장되는 디렉터리
dataset_root = "/content/drive/MyDrive/SBV2-KR/Data"

# 학습 결과(음성 합성에 필요한 파일)가 저장되는 디렉터리
assets_root = "/content/drive/MyDrive/SBV2-KR/model_assets"

import yaml


with open("configs/paths.yml", "w", encoding="utf-8") as f:
    yaml.dump({"dataset_root": dataset_root, "assets_root": assets_root}, f)

## 2. 학습 데이터 준비

음성 파일(파일당 2~12초 정도)과 전사 데이터가 이미 있다면 2.2를, 없다면 2.1을 실행하세요.


### 2.1 음성 파일로 데이터셋 만들기 (전사가 있으면 건너뛰기)

음성 파일과 전사 데이터셋이 없다면 한국어 음성 파일만으로 데이터셋을 만들 수 있습니다. Google Drive의 `SBV2-KR/inputs/` 폴더에 음성 파일(wav·mp3 등 일반 형식, 하나든 여러 개든 가능)을 넣고 아래 셀을 실행하면, 음성을 자르고 Whisper로 전사해 데이터셋을 올바른 위치에 만듭니다.

`inputs/` 안에 하위 폴더를 2개 이상 만들고 스타일별로 음성 파일을 나눠 넣으면, 학습할 때 하위 폴더에 따라 스타일이 자동으로 만들어집니다. 기본 스타일만 쓰거나 나중에 직접 스타일을 만들 거라면 `inputs/` 바로 아래에 넣으면 됩니다.


In [ ]:
# 원본 음성 파일을 넣은 디렉터리
input_dir = "/content/drive/MyDrive/SBV2-KR/inputs"
# 모델명(화자명)
model_name = "your_model_name"

# 이렇게 전사해 달라는 예문 (문장부호 사용법·웃음 표기·고유명사 등)
initial_prompt = "안녕하세요. 잘 지내시나요? 후훗, 저는…… 잘 지내고 있어요!"

!python slice.py -i {input_dir} --model_name {model_name}
!python transcribe.py --model_name {model_name} --language ko --initial_prompt "{initial_prompt}" --use_hf_whisper

성공하면 그대로 3으로 진행하세요.


### 2.2 음성 파일과 전사 데이터가 이미 있을 때

안내에 따라 데이터셋을 배치하세요.

다음 셀을 실행해 학습 데이터를 넣을 폴더(1에서 설정한 `dataset_root`)를 만듭니다.


In [ ]:
import os

os.makedirs(dataset_root, exist_ok=True)

음성 데이터와 전사 텍스트를 준비해 다음과 같이 배치합니다.
```
├── Data/
│   ├── {모델 이름}
│   │   ├── esd.list
│   │   ├── raw/
│   │   │   ├── foo.wav
│   │   │   ├── bar.mp3
│   │   │   ├── style1/
│   │   │   │   ├── baz.wav
│   │   │   │   ├── qux.wav
│   │   │   ├── style2/
│   │   │   │   ├── corge.wav
│   │   │   │   ├── grault.wav
...
```

### 배치 방법
- 위와 같이 배치하면 `style1/`과 `style2/` 폴더 내부(바로 아래 외의 위치 포함)에 들어 있는 음성 파일들로부터, 기본 스타일에 더해 `style1`과 `style2`라는 스타일이 자동으로 생성됩니다.
- 딱히 스타일을 만들 필요가 없거나 스타일 분류 기능 등으로 스타일을 만드는 경우, `raw/` 폴더 바로 아래에 전부 배치해 주세요. 이렇게 `raw/`의 하위 디렉터리 개수가 0 또는 1인 경우, 스타일은 기본 스타일만 생성됩니다.
- 음성 파일 포맷은 wav 형식 외에도 mp3 등 다양한 음성 파일에 대응합니다.

### 전사 파일 `esd.list`

`Data/{모델 이름}/esd.list` 파일에는 다음 포맷으로 각 음성 파일의 정보를 기술해 주세요.

```
path/to/audio.wav(wav 파일이 아니어도 이렇게 기재)|{화자명}|{언어 ID, 한국어는 KO}|{전사 텍스트}
```

- 여기서 첫 번째 `path/to/audio.wav`는 `raw/` 기준 상대 경로입니다. 즉 `raw/foo.wav`의 경우 `foo.wav`, `raw/style1/bar.wav`의 경우 `style1/bar.wav`가 됩니다.
- 확장자가 wav가 아닌 경우에도 `esd.list`에는 `wav`로 적어 주세요. 즉 `raw/bar.mp3`의 경우에도 `bar.wav`로 적어 주세요.

예:
```
foo.wav|hanako|KO|안녕하세요, 잘 지내세요?
bar.wav|taro|KO|네, 들리고 있어요……. 무슨 일이세요?
style1/baz.wav|hanako|KO|오늘은 날씨가 좋네요.
style1/qux.wav|taro|KO|네, 그러네요.
...
```
물론 화자 한 명의 데이터셋이어도 괜찮습니다.


## 3. 학습 전처리

학습 전처리를 진행합니다. 필요한 설정을 다음 셀에 입력하고 실행하세요. 「~할지」 항목에는 `True` 또는 `False`를 지정합니다.


In [ ]:
# 위에서 정한 폴더 이름 `Data/{model_name}/`
model_name = "your_model_name"

# 학습 배치 크기. VRAM에 맞춰 조정하세요. (VRAM 사용량 기준 1: 6GB, 2: 8GB, 3: 10GB, 4: 12GB)
batch_size = 4

# 학습 에폭 수 (데이터셋을 몇 바퀴 돌지).
# 100은 많은 편일 수 있지만, 더 많이 하면 품질이 오를 수도 있습니다.
epochs = 100

# 저장 주기. 몇 스텝마다 모델을 저장할지. 잘 모르겠으면 기본값 그대로 두세요.
save_every_steps = 1000

# 음성 파일의 음량을 정규화할지
normalize = False

# 음성 파일 시작·끝의 무음 구간을 잘라낼지
trim = False

# 읽기 오류가 난 줄을 어떻게 할지.
# "raise"는 텍스트 전처리가 끝난 뒤 중단, "skip"은 읽지 못한 줄을 학습에서 제외, "use"는 그대로 사용
yomi_error = "skip"

위 셀을 실행했다면 다음 셀로 학습 전처리를 진행합니다.


In [ ]:
from gradio_tabs.train import preprocess_all
from style_bert_vits2.nlp.japanese import pyopenjtalk_worker


pyopenjtalk_worker.initialize_worker()

preprocess_all(
    model_name=model_name,
    batch_size=batch_size,
    epochs=epochs,
    save_every_steps=save_every_steps,
    num_processes=2,
    normalize=normalize,
    trim=trim,
    freeze_EN_bert=False,
    freeze_JP_bert=False,
    freeze_ZH_bert=False,
    freeze_style=False,
    freeze_decoder=False,
    use_jp_extra=True,  # 한국어는 JP-Extra 아키텍처로 학습한다
    val_per_lang=0,
    log_interval=200,
    yomi_error=yomi_error,
)

## 4. 학습

전처리가 정상적으로 끝나면 학습을 진행합니다. 다음 셀들을 실행하면 학습이 시작됩니다.

학습은 일본어 사전학습 모델(JP-Extra 베이스)에서 시작하며, 한국어 음소 임베딩은 발음이 비슷한 일본어 음소로 자동 초기화(warm-start)됩니다.

학습 결과는 위에서 지정한 `save_every_steps` 간격으로 Google Drive의 `SBV2-KR/model_assets/{모델 이름}/` 폴더에 저장됩니다.

이 폴더를 내려받아 로컬 SBV2-KR의 `model_assets` 폴더에 넣으면 학습 결과를 쓸 수 있습니다.


In [ ]:
# 위에서 정한 모델명. 이어서 학습할 때는 모델이 저장된 폴더 이름을 정확히 입력하세요.
model_name = "your_model_name"


import yaml
from gradio_tabs.train import get_path

paths = get_path(model_name)
dataset_path = str(paths.dataset_path)
config_path = str(paths.config_path)

with open("default_config.yml", "r", encoding="utf-8") as f:
    yml_data = yaml.safe_load(f)
yml_data["model_name"] = model_name
with open("config.yml", "w", encoding="utf-8") as f:
    yaml.dump(yml_data, f, allow_unicode=True)

In [ ]:
!python train_ms_jp_extra.py --config {config_path} --model {dataset_path} --assets_root {assets_root}

## 5. 학습 결과 확인

WebUI에서 학습한 모델로 음성을 합성해 보거나, 모델 병합·스타일 나누기를 할 수 있습니다.


In [ ]:
!python app.py --share